# End-to-end brain network analysis pipeline

This notebook demonstrates the full three-section pipeline for analysing
signed fMRI correlation matrices:

1. **Descriptive analysis** of the raw signed dense network
2. **Network filtering** to produce tractable unsigned networks
3. **Standard metrics + LRG multiscale analysis** on filtered networks

Replace the synthetic data with your own correlation matrices to use this
as a template.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from multifunbrain.pipeline import run_pipeline, PipelineConfig
from multifunbrain.io import load_correlation_matrix
from multifunbrain.preprocessing import prepare_correlation_matrix
from multifunbrain.analysis.descriptive import (
    weight_distribution_analysis,
    correlation_spectrum_analysis,
    signed_laplacian_analysis,
    signed_network_metrics,
)
from multifunbrain.processing import apply_all_filters
from multifunbrain.analysis.network import network_summary_report

## Step 1: Create or load a correlation matrix

Here we generate a synthetic signed correlation matrix with modular
structure.  Replace this with `load_correlation_matrix('path/to/matrix.npy')`
for real data.

In [ ]:
rng = np.random.default_rng(42)
n_regions = 48
n_timepoints = 300

# Create modular time series
n_modules = 4
module_size = n_regions // n_modules
ts = rng.normal(size=(n_regions, n_timepoints))

# Add within-module correlations
for m in range(n_modules):
    shared = rng.normal(size=n_timepoints)
    start = m * module_size
    for i in range(start, start + module_size):
        ts[i] += 0.5 * shared

corr = np.corrcoef(ts)
corr_clean = prepare_correlation_matrix(corr)

print(f"Matrix shape: {corr_clean.shape}")
print(f"Value range: [{corr_clean.min():.3f}, {corr_clean.max():.3f}]")

## Section 1: Descriptive analysis of the raw signed network

### 1a. Weight distribution

In [ ]:
wd = weight_distribution_analysis(corr_clean)

print(f"Positive edges: {wd['n_positive']} ({wd['frac_positive']:.1%})")
print(f"Negative edges: {wd['n_negative']} ({wd['frac_negative']:.1%})")
print(f"Mean weight: {wd['mean']:.4f}, Std: {wd['std']:.4f}")
print(f"Skewness: {wd['skewness']:.3f}, Kurtosis: {wd['kurtosis']:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(wd['all_weights'], bins=80, edgecolor='black', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', label='zero')
ax.set_xlabel('Correlation weight')
ax.set_ylabel('Count')
ax.set_title('Edge weight distribution (signed correlation network)')
ax.legend()
plt.tight_layout()
plt.show()

### 1b. Eigenvalue spectrum and RMT validation

In [ ]:
gamma = n_regions / n_timepoints
spec = correlation_spectrum_analysis(corr_clean, gamma=gamma)

print(f"Aspect ratio (gamma): {gamma:.3f}")
print(f"MP bounds: [{spec['mp_lambda_minus']:.4f}, {spec['mp_lambda_plus']:.4f}]")
print(f"Signal eigenvalues: {spec['n_signal']}")
print(f"Noise eigenvalues: {spec['n_noise']}")
print(f"Largest eigenvalue: {spec['largest_eigenvalue']:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(spec['eigenvalues'], bins=40, density=True, alpha=0.7, label='Empirical')
ax.axvline(spec['mp_lambda_plus'], color='red', linestyle='--', label=f"MP upper = {spec['mp_lambda_plus']:.3f}")
ax.axvline(spec['mp_lambda_minus'], color='red', linestyle=':', label=f"MP lower = {spec['mp_lambda_minus']:.3f}")
ax.set_xlabel('Eigenvalue')
ax.set_ylabel('Density')
ax.set_title('Correlation matrix spectrum vs Marchenko-Pastur')
ax.legend()
plt.tight_layout()
plt.show()

### 1c. Signed Laplacian analysis

In [ ]:
sl = signed_laplacian_analysis(corr_clean, n_modes=6)

print(f"Negative eigenvalues: {sl['n_negative_eigenvalues']}")
print(f"Frustration index: {sl['frustration_index']:.4f}")
print(f"Spectral gap: {sl['spectral_gap']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(sl['eigenvalues'], 'o-', markersize=3)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Index')
axes[0].set_ylabel('Eigenvalue')
axes[0].set_title('Signed Laplacian spectrum')

# First few eigenmodes
for i in range(min(4, sl['leading_modes'].shape[1])):
    axes[1].plot(sl['leading_modes'][:, i], label=f'Mode {i}')
axes[1].set_xlabel('Node index')
axes[1].set_ylabel('Component value')
axes[1].set_title('Leading signed Laplacian eigenmodes')
axes[1].legend()

plt.tight_layout()
plt.show()

### 1d. Signed network metrics

In [ ]:
nm = signed_network_metrics(corr_clean)

print(f"Nodes: {nm['n_nodes']}")
print(f"Edges: {nm['n_edges']} / {nm['n_possible_edges']}")
print(f"Density: {nm['density']:.3f}")
print(f"Balance ratio (pos / total): {nm['balance_ratio']:.3f}")
print(f"Mean strength: {nm['mean_strength']:.3f} +/- {nm['std_strength']:.3f}")

## Section 2: Network filtering

Apply different methods to convert the signed dense matrix into
tractable unsigned networks.

In [ ]:
filters = apply_all_filters(
    corr_clean,
    methods=['absolute', 'positive', 'negative', 'disparity', 'mp_validated'],
    threshold=0.0,
    alpha=0.05,
    gamma=gamma,
)

print(f"{'Filter':<25} {'Nodes':>6} {'Edges':>7} {'Density':>8}")
print('-' * 50)
for name, data in filters.items():
    G = data['graph']
    n, m = G.number_of_nodes(), G.number_of_edges()
    d = 2 * m / (n * (n - 1)) if n > 1 else 0
    print(f"{name:<25} {n:>6} {m:>7} {d:>8.3f}")

## Section 3: Standard metrics + LRG on filtered networks

### 3a. Full pipeline (all sections at once)

In [ ]:
config = PipelineConfig(
    gamma=gamma,
    filter_methods=['absolute', 'positive', 'mp_validated'],
    run_lrg=True,
    run_community_detection=True,
    run_rich_club=False,
    seed=42,
)

result = run_pipeline(corr_clean, config=config, label='synthetic_48regions')

print("Summary table:")
print(result.summary_table().to_string(index=False))

### 3b. Inspect LRG partitions

In [ ]:
for fname, parts in result.lrg_results.items():
    print(f"\nFilter: {fname}")
    if isinstance(parts, dict) and "error" in parts:
        print(f"  LRG failed: {parts['error']}")
        print(f"  ({parts['n_nodes']} nodes, {parts['n_edges']} edges)")
        continue
    for p in parts:
        n_clusters = len(np.unique(p['partition']))
        print(f"  tau={p['tau']:.3f} -> {n_clusters} clusters")

### 3c. Using individual functions

You can also use each function independently:

In [ ]:
from multifunbrain.analysis.network import (
    compute_global_metrics,
    compute_node_metrics,
    detect_communities_louvain,
)

# Pick one filtered network
G = result.filtered_networks['positive']['graph']

# Global metrics
gm = compute_global_metrics(G)
for k, v in gm.items():
    print(f"  {k}: {v}")

# Community detection
partition = detect_communities_louvain(G, seed=42)
print(f"\nLouvain communities: {len(set(partition.values()))}")

# Node metrics with community info
node_df = compute_node_metrics(G, community_partition=partition)
print(f"\nNode metrics (top 5 by betweenness):")
print(node_df.sort_values('betweenness', ascending=False).head())

## Batch analysis

Process multiple matrices and compare:

In [ ]:
from multifunbrain.pipeline import run_pipeline_batch

# Generate 3 different correlation matrices
matrices = []
labels = []
for i in range(3):
    rng_i = np.random.default_rng(i)
    ts_i = rng_i.normal(size=(48, 300))
    for m in range(4):
        shared = rng_i.normal(size=300) * (0.3 + 0.2 * i)
        for j in range(m * 12, (m + 1) * 12):
            ts_i[j] += shared
    matrices.append(np.corrcoef(ts_i))
    labels.append(f'subject_{i}')

config_batch = PipelineConfig(
    gamma=48 / 300,
    filter_methods=['positive'],
    run_lrg=False,
    seed=42,
)

results = run_pipeline_batch(matrices, config=config_batch, labels=labels)

import pandas as pd
summary = pd.concat([r.summary_table() for r in results], ignore_index=True)
print(summary.to_string(index=False))